In [2]:
import pandas as pd
import fastparquet
import openpyxl
import os
import numpy as np
from geopy.distance import geodesic
from geopy.geocoders import Nominatim
from datetime import datetime, time, date

In [10]:
hr_mapping = {
    'id' : np.int32 ,
    'last_name'  :object,
    'first_name' : object,
    'birthday' : None,
    'business_unit' : object,
    'entry_date' : None,
    'salary' : np.float64,
    'employement contract' :object,
    'vacation_days': np.int32,
    'address' : object,
    'transport_mode' : object,
    'choice' : object
}
hr_names = hr_mapping.keys()

In [11]:
df_rh = pd.read_excel("../sources/Fusion.xlsx", names= list(hr_names), dtype= hr_mapping  )
df_rh

,id,last_name,first_name,birthday,business_unit,entry_date,salary,employement contract,vacation_days,address,transport_mode,choice
0,86847,Arnaud,Jeremy,1989-05-13,Support,2020-03-31,29180.0,CDI,28,"139 Av. des Pins, 34150 Gignac",véhicule thermique/électrique,Football
1,92368,Torres,Emmanuelle,2001-06-25,Marketing,2020-04-21,74310.0,CDI,28,"Pl. de la Liberté, 34470 Pérols",Vélo/Trottinette/Autres,Runing
2,91916,Legrand,Margaud,1983-10-13,Finance,2020-05-19,47530.0,CDI,27,"72 Rue des Pins, 34470 Pérols",Vélo/Trottinette/Autres,Rugby
3,36913,Pons,Brigitte,1977-12-27,Finance,2020-05-26,59790.0,CDI,28,"169 Rue Jean Jaures, 34730 Prades-le-Lez",Transports en commun,0
4,79837,Muller,Alphonse,1970-12-10,Ventes,2020-05-26,30360.0,CDI,27,"11 Lot. Mas de Combes, 34230 Popian",véhicule thermique/électrique,0
...,...,...,...,...,...,...,...,...,...,...,...,...
156,18941,Guillou,Gilles,1986-12-23,Finance,2025-02-27,59910.0,CDI,29,"68 Bd des Écoles, 34750 Villeneuve-lès-Maguelone",Vélo/Trottinette/Autres,Runing
157,41377,Faure,Emmanuelle,1998-06-16,Support,2025-03-05,52920.0,CDI,25,"Rue du Ctre, 34160 Saint-Drézéry",véhicule thermique/électrique,Football
158,43015,Mendes,Juliette,1991-10-22,Support,2025-03-06,57040.0,CDI,26,"199-97 Av. du Pichagret, 34980 Saint-Gély-du-Fesc",véhicule thermique/électrique,0
159,19773,Fischer,Benoit,1981-03-31,Support,2025-03-24,62670.0,CDI,25,"110 Rue de la République, 34000 Montpellier",véhicule thermique/électrique,0


In [62]:
df_rh.dtypes

id                               int32
last_name                       object
first_name                      object
birthday                datetime64[us]
business_unit                   object
entry_date              datetime64[us]
salary                         float64
employement contract            object
vacation_days                    int32
address                         object
transport_mode                  object
dtype: object

,id,last_name,first_name,birthday,business_unit,entry_date,salary,employement contract,vacation_days,address,transport_mode
0,59019,Colin,Audrey,1990-07-06,Marketing,2020-12-14,30940.0,CDI,29,"128 Rue du Port, 34000 Frontignan",Transports en commun
1,19841,Ledoux,Monique,1962-01-06,R&D,2020-07-07,74360.0,CDI,26,"68 Rue du Port, 34970 Saint-Clément-de-Rivière",véhicule thermique/électrique
2,56482,Dumont,Michelle,1976-08-09,Ventes,2022-03-29,51390.0,CDI,27,"100 Av. de la Gare, 30900 Nîmes",véhicule thermique/électrique
3,21886,Toussaint,Judith,1962-09-10,Support,2021-12-12,70320.0,CDI,29,"53 Av. de la Gare, 34970 Lattes",Marche/running
4,81001,Bailly,Michelle,1975-04-20,Ventes,2025-02-19,46870.0,CDD,29,"74 Rue des Fleurs, 34970 Lattes",Marche/running
...,...,...,...,...,...,...,...,...,...,...,...
156,18941,Guillou,Gilles,1986-12-23,Finance,2025-02-27,59910.0,CDI,29,"68 Bd des Écoles, 34750 Villeneuve-lès-Maguelone",Vélo/Trottinette/Autres
157,81676,Breton,Jeannine,1966-06-24,Ventes,2022-01-22,48430.0,CDI,29,"27 Av. du Général Leclerc, 34470 Pérols",Marche/running
158,27069,Delahaye,Philippe,1996-09-10,Support,2020-11-27,56380.0,CDI,29,"59 Chem. des Pins, 34170 Castelnau-le-Lez",véhicule thermique/électrique
159,30256,Dumas,Odette,1960-12-22,Ventes,2023-04-08,45170.0,CDI,29,"2 Rue du Nord, 34920 Le Crès",véhicule thermique/électrique


In [5]:
print("--- Analyse géographique des déplacements ---")
geolocator = Nominatim(user_agent="sport_pipeline_poc")
company_address = "1362 Av. des Platanes, 34970 Lattes"
location_comp = geolocator.geocode(company_address)
coords_company = (location_comp.latitude, location_comp.longitude)
coords_company

--- Analyse géographique des déplacements ---


(43.5873166, 3.9180516)

In [5]:
hr_proc = pd.read_parquet("../kestra/tmp/hr_processed.parquet")
hr_proc

,id,last_name,first_name,business_unit,salary,employement_contract,vacation_days,transport_mode,distance_kms,age,seniority_years,margin_kms
0,NaN,gAAAAABpvsu9tvUPEwJ6DBqK84vcRa5dzxFbYmPNKIzlgh...,gAAAAABpvsu9TcoYNT6fukWumMT4as-PDXCBHg9i3wwazc...,Marketing,30940.0,CDI,29,Transports en commun,0.0,35,5,150.0
1,19841.0,gAAAAABpvsu9gh5yN4ZIrr3W1DqqdEBSLZRJKW3PVVwjVw...,gAAAAABpvsu91yqCQAd7h_MOXzV0cYPqBzeUHEvzf1acS1...,R&D,74360.0,CDI,26,véhicule thermique/électrique,0.0,64,5,250.0
2,56482.0,gAAAAABpvsu9Ryw1m-Xy6q7c4u_d1iZVQjW7H26wGvoSgh...,gAAAAABpvsu9xf_5ugntY0FUJPWZUvjlDjAVY-xPMgLwB6...,Ventes,51390.0,CDI,27,véhicule thermique/électrique,0.0,49,3,250.0
3,21886.0,gAAAAABpvsu9DGVkm40a71MS3VPDfIkJcicSv9kn33zsCt...,gAAAAABpvsu9_-uOzGz7FqgR0-gFtIm-T2KX4AP3zVI17-...,Support,70320.0,CDI,29,Marche/running,0.0,63,4,15.0
4,81001.0,gAAAAABpvsu9L8cEETqWmXiqisyK5dq3yI9-QKHCG9O9O_...,gAAAAABpvsu92HGS08iaNdKIRpBWocDR4UU4kiafPeBPRZ...,Ventes,46870.0,CDD,29,Marche/running,0.0,50,1,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...
156,18941.0,gAAAAABpvsu9PZL50Q0DFt399Gagv5WiwK68qHE9Q2vIgc...,gAAAAABpvsu9s_-b-_uFFMF_Jaw6VLtj1sbfVXKilMuCtu...,Finance,59910.0,CDI,29,Vélo/Trottinette/Autres,0.0,39,1,25.0
157,81676.0,gAAAAABpvsu9NKmK3POrzxEDCXUJqDyEvymAyzY1tWJFVy...,gAAAAABpvsu94HXFLIOcvzB9xrgcUk84LEBqEtUHnYL3Ea...,Ventes,48430.0,CDI,29,Marche/running,0.0,59,4,15.0
158,27069.0,gAAAAABpvsu9xpkfuzkvvx2UULdQ8EWIreXCcCyLYVliav...,gAAAAABpvsu9TLS8X_4__w-bOIsoRad0oYdt09GiH-gUfF...,Support,56380.0,CDI,29,véhicule thermique/électrique,0.0,29,5,250.0
159,30256.0,gAAAAABpvsu9-mGpisfHXlv8ZIQnVFbyjjRnNjCX_uUvbq...,gAAAAABpvsu9YE9Ypxfh3Y03JeDlBfVU2wSAPRhN9bj7h8...,Ventes,45170.0,CDI,29,véhicule thermique/électrique,0.0,65,2,250.0


In [ ]:
#df_sport = pd.read_excel("../sources/Fusion.xlsx", sheet_name= "Feuille2", names=['id',"sport_type"])
df_sport = pd.read_excel("../sources/Fusion.xlsx", sheet_name= "Feuille2", names=['id',"sport_type"])
df_sport

,id,sport_type
0,12497,NaN
1,13064,NaN
2,13780,NaN
3,15493,NaN
4,16872,NaN
...,...,...
156,96558,Triathlon
157,96902,NaN
158,97309,Tennis
159,99401,Runing


In [3]:
df_merge = pd.read_parquet("../kestra/tmp/hr_raw.parquet")
df_merge

NameError: name 'pd' is not defined

In [26]:
df_sport['sport_type'].notna().sum()
df_test = df_sport.merge(df_merge, how= 'left', on="id", indicator=True)
df_test

,id,sport_type,last_name,first_name,birthday,business_unit,entry_date,salary,employement contract,vacation_days,address,transport_mode,distance_kms,margin_kms,_merge
0,12497,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,13064,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,13780,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,15493,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,16872,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,96558,Triathlon,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
157,96902,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
158,97309,Tennis,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
159,99401,Runing,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [27]:
df_extra_sports = df_test[df_test['_merge'] == 'left_only']
df_extra_sports

,id,sport_type,last_name,first_name,birthday,business_unit,entry_date,salary,employement contract,vacation_days,address,transport_mode,distance_kms,margin_kms,_merge
0,12497,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,13064,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,13780,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,15493,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,16872,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,96558,Triathlon,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
157,96902,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
158,97309,Tennis,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
159,99401,Runing,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [ ]:
df_extra_sports[]

np.int64(96558)

In [34]:
df_extra_declared_sports = df_extra_sports[df_extra_sports['sport_type'].notnull()]
df_extra_declared_sports.columns.to_list()

['id',
 'sport_type',
 'last_name',
 'first_name',
 'birthday',
 'business_unit',
 'entry_date',
 'salary',
 'employement contract',
 'vacation_days',
 'address',
 'transport_mode',
 'distance_kms',
 'margin_kms',
 '_merge']

In [3]:
sports = pd.read_excel('../data/sources/strava_sports.xlsx')
sports

,sport,strava_list,min_distance,max_distance,min_lasting,max_lasting,popularity_score,alias
0,Course à pied,1,2000,21000,15,150,100,"running, running"
1,Trail,1,5000,30000,45,240,60,NaN
2,Marche,1,1000,10000,20,120,95,NaN
3,Randonnée,1,3000,25000,60,480,80,NaN
4,Course à pied virtuelle,1,2000,15000,15,90,10,NaN
5,Vélo,1,5000,100000,30,300,100,NaN
6,Sortie en VTT,1,5000,50000,45,240,75,NaN
7,Sortie en gravel,1,10000,80000,60,300,30,NaN
8,Sortie à vélo électrique,1,10000,60000,30,180,50,NaN
9,VTT électrique,1,10000,50000,45,240,40,NaN


In [14]:
tab = [{row['sport'] : row['alias'].strip().split(',')} for _, row in sports.iterrows() if isinstance(row['alias'],str)]
tab

[{'Course à pied': ['running', ' running']}]

In [52]:
from enum import Enum
class Ttype(Enum):
    TC = 'Transports en commun'
    VM = 'véhicule thermique/électrique'
    MR = 'Marche/running'
    VT = 'Vélo/Trottinette/Autres' 

TRANSPORT_LIMIT = {
    Ttype.TC : 0,
    Ttype.VM : 0,
    Ttype.MR : 15,
    Ttype.VT : 25
}

In [59]:
dir(Ttype)

['MR',
 'TC',
 'VM',
 'VT',
 '__class__',
 '__contains__',
 '__doc__',
 '__getitem__',
 '__init_subclass__',
 '__iter__',
 '__len__',
 '__members__',
 '__module__',
 '__name__',
 '__qualname__']

In [8]:
sports.index

RangeIndex(start=0, stop=54, step=1)

In [1]:
from cryptography.fernet import Fernet

In [4]:
CRYPT_KEY = Fernet.generate_key().decode()
CRYPT_KEY

'rHhARZ_cZ30HT_MRFyOxj99N-IZOXl6Spa4tJKiSvPA='

In [5]:
cipher_suite = Fernet(CRYPT_KEY.encode())
cipher_suite.encrypt(str('').encode())

b'gAAAAABptUB5bzeHCOn1DxJZSjm51zRXIGGTEjZR-3QbhLb_e91KViTR3fME6ydc13AgjAkLLk3GXmoosy249JeHsOEkHtlnTQ=='

In [19]:
multi_ratio = 2
total = 3000



nb1 = 93
nb3 = 68
ratio = ( multi_ratio * nb3 )/(multi_ratio * nb3 + nb1)
sample1 = int((1-ratio) * total)
print(ratio)
print(sample1)
scale =70
poids1 = np.sort(np.random.exponential(scale=scale, size=nb1))[::-1]
poids_norm1 = (poids1 / poids1.sum())*sample1
print(poids_norm1)



0.5938864628820961
1218
[41.30424186 39.06658056 38.8837282  35.73310686 35.07681831 34.31759546
 34.16384429 32.5829946  32.18105596 30.98875317 30.6634237  29.85219459
 29.81056094 27.02079168 26.68500286 26.28014936 25.32310522 23.57288201
 22.69029838 20.10069633 19.82401209 19.50795286 18.38309809 18.37063739
 18.03119581 16.36801968 15.94623911 15.75834984 15.68214852 15.6701342
 15.62628591 15.6199265  15.14176712 13.51724216 12.78225767 12.57590857
 12.18402695 11.62241147 11.50281964 11.38821966 11.37992536 10.34599293
 10.31492952 10.26354718  9.94154209  9.83106154  9.77159878  9.76410499
  9.37769917  9.19273945  8.85365872  8.39309808  8.36456528  8.2874474
  7.93704474  7.9023525   7.70959004  7.36774775  7.25704415  6.78098418
  6.63411853  6.34298474  6.00825838  5.97801078  5.94906378  5.82815571
  5.72423431  5.50421498  5.49695551  5.14655848  5.1159017   5.01404952
  4.99690607  4.37405756  4.08545565  3.89401987  3.8110252   3.77703598
  2.81666201  2.8005764   2.3

In [19]:
sample2 = sample1
nb2 = 93
size = 0.9
poids2 = np.random.lognormal(mean=0, sigma=size, size=nb2)
poids_norm2 = ( poids2 / poids2.sum()) * sample2
list(poids_norm2).sort(reverse=True)
poids_norm2

array([ 6.7094328 ,  3.50558756, 33.04649066, 30.76190845, 10.3021653 ,
       45.30617374,  4.97683371, 10.16038525,  4.45090148,  7.00316999,
        6.63225395,  4.82432988, 25.04563625, 11.40943631, 36.14941143,
       36.92219962,  7.80773104,  7.37293   ,  2.63001894, 11.35748395,
        1.14770135, 16.21179376,  8.7841294 , 11.14215368,  5.88683463,
       11.4045483 , 10.63175239,  5.57409105,  7.04070445,  6.69138834,
       54.7943345 , 10.43618065, 66.62711985,  7.68001408,  6.29440101,
        6.54157183, 11.54841075, 11.49123179, 25.21091056, 25.02502946,
       38.87725383, 25.72204972, 25.57464029, 17.16018807,  6.4851658 ,
       16.17984399, 15.57644939,  3.89394047,  2.16327007, 15.93140288,
       14.20817173,  9.74042873,  9.02004348,  4.51506561,  1.5996326 ,
       14.80031458, 14.29454444,  4.18265289,  0.81560537,  7.38468546,
        4.64809994, 13.47123164,  2.12472898, 17.34255795,  6.97759301,
       17.67935757, 16.64514516, 24.32711995, 23.95860021, 11.24

In [ ]:

sample3 = total - sample1
print(sample3)

pow3 = 0.4
rangs3 = np.arange(1, nb3 + 1)
# On inverse et on applique une puissance faible (0.5 = racine carrée)
poids3 = 1 / np.power(rangs3, pow3) 
np.random.shuffle(poids3) # Pour ne pas que ce soit toujours les mêmes IDs
poids_norm3 = (poids3 / poids3.sum())*sample3
print(poids_norm3)

1651
[15.90717577 40.48454434 20.98964427 17.33651318 16.1171874  28.84685965
 22.87566628 18.41460356 15.51429987 23.6514963  19.14802179 28.06165314
 19.99505201 30.68154727 17.19973221 16.22590583 25.01138593 18.58874371
 29.71477286 24.52799437 16.01097656 16.45128452 38.0636595  17.77094868
 17.92448238 25.52985307 47.61302334 43.54735222 18.08276133 19.77100543
 20.22824425 36.08392413 22.51958675 21.26675627 34.42331983 15.80569277
 15.70644024 33.00272159 23.25226474 17.62192144 21.55711093 20.72477214
 15.42125912 22.18218128 16.81094982 82.89908857 26.69132831 20.47124141
 27.34650079 17.47717858 16.93711389 16.33723508 31.76820538 18.24604263
 16.68801494 26.08799807 62.82576095 19.55550637 21.86183153 17.06665535
 19.34801062 16.56817011 18.76878732 24.07579759 18.955086   15.60933542
 15.33014246 53.41967652]


In [26]:
pow4 = 0.9
rangs4 = np.arange(1, nb1 + 1)
# On inverse et on applique une puissance faible (0.5 = racine carrée)
poids4 = 1 / np.power(rangs4, pow4) 
np.random.shuffle(poids4) # Pour ne pas que ce soit toujours les mêmes IDs
poids_norm4 = (poids4 / poids4.sum())*sample1
print(poids_norm4)

[  4.66886268   3.9623057    9.44418218 114.51762704   4.55197732
   7.55603334   9.18621891   4.60963802  29.57883589   5.70729225
   5.44543434   5.28448275   4.49581842  26.90291481   4.38777348
  12.23514777   5.36368434   4.23559273  10.65018393   5.06101192
   4.28506766   5.52986007   3.65081891  11.0045414    5.99723482
   7.90389551  79.50422476   3.83912285  10.00898889   6.32013467
   8.09084906   7.09074598   5.13325504   6.55665334   4.14017166
   3.92032709   4.99088216   6.68207374  12.71288985  24.69141164
  42.60525913  18.67743369  37.08609448  14.41691508   4.8565964
  11.38474541   8.94269278   4.00524469   4.92277282   4.04917801
  17.6234671   13.23179987   3.87927645   4.72971699   8.28739015
  13.79754983   7.7258335    6.4361012    4.09414123   9.71793102
   3.7613897    7.39392388   5.89718772   3.61546934  21.24468614
   5.20771029  15.09805821   6.100934     6.81266898   4.33577866
   4.7922704    5.61709755   6.94877219  19.87394595 213.69744829
   3.723754

In [3]:
df_activities = pd.read_parquet("../kestra/tmp/activities.parquet")

df_activities

,employee_id,sport,distance_meters,begin_date,duration_sec,id
0,69938,Randonnée,5701,2025-09-14 17:34:00,6597,ACT-20250914-173400-842
1,89328,Tennis de table,0,2025-12-20 12:03:00,3246,ACT-20251220-120300-613
2,41377,Football,4967,2025-11-23 08:23:00,3725,ACT-20251123-082300-114
3,42960,Runing,13136,2025-11-14 12:24:00,5654,ACT-20251114-122400-030
4,29450,Surf,0,2026-03-21 09:44:00,10760,ACT-20260321-094400-761
5,36827,Tennis,0,2025-12-18 12:05:00,4409,ACT-20251218-120500-888
6,76076,Football,6372,2025-09-13 09:05:00,4779,ACT-20250913-090500-599
7,33386,Basketball,0,2025-09-13 14:16:00,5854,ACT-20250913-141600-695
8,41377,Course à pied virtuelle,9285,2025-11-12 06:01:00,3441,ACT-20251112-060100-694
9,67121,Rugby,4662,2026-06-20 18:20:00,5284,ACT-20260620-182000-087


In [4]:
df_activities.to_excel("../kestra/tmp/activities.xlsx")

In [21]:
datetime.fromisoformat("2026-01-01")

datetime.datetime(2026, 1, 1, 0, 0)

In [26]:
[i for i in range(5,32,7)]


[5, 12, 19, 26]

In [31]:
dt = datetime(2026,2,13,12,21)
dt

datetime.datetime(2026, 2, 13, 12, 21)

In [33]:
ts = dt.timestamp()
ts

1770981660.0

In [46]:
dt2 = datetime.fromtimestamp(1765092782.0)
dt2

datetime.datetime(2025, 12, 7, 8, 33, 2)

In [61]:
class Ttype(Enum):
    @classmethod
    def values(cls):
        return [member.value for member in cls]
    TC = 'Transports en commun'
    VM = 'véhicule thermique/électrique'
    MR = 'Marche/running'
    VT = 'Vélo/Trottinette/Autres' 

TRANSPORT_LIMIT = {
    Ttype.TC.value : 150,
    Ttype.VM.value : 150,
    Ttype.MR.value : 15,
    Ttype.VT.value : 25
}

In [74]:
SP2 = " "*2
SP4 = " "*4

In [79]:
a = f"{SP2}\n{SP4}- " + f"{SP2}\n{SP4}- ".join([f"{index} <= {TRANSPORT_LIMIT[index]}kms" for index in TRANSPORT_LIMIT.keys()])+ f"{SP2}\n"
print(a)

  
    - Transports en commun <= 150kms  
    - véhicule thermique/électrique <= 150kms  
    - Marche/running <= 15kms  
    - Vélo/Trottinette/Autres <= 25kms  



In [12]:
import requests
import pandas as pd

def download_herault_spots_full():
    overpass_url = "http://overpass-api.de/api/interpreter"
    
    # On cherche (node, way, relation) avec nwr
    # On cible plus large : sport, loisirs, parcs, plages, sommets
    overpass_query = """
    [out:json][timeout:180];
    area(3600007411)->.searchArea;
    (
      nwr["sport"](area.searchArea);
      nwr["leisure"~"park|pitch|garden|beach_resort|nature_reserve|marina"](area.searchArea);
      nwr["tourism"~"viewpoint|picnic_site"](area.searchArea);
      nwr["natural"~"peak|beach|water"](area.searchArea);
    );
    out center;
    """
    
    try:
        response = requests.get(overpass_url, params={'data': overpass_query}, timeout=200)
        response.raise_for_status()
        data = response.json()

        spots = []
        for element in data.get('elements', []):
            tags = element.get('tags', {})
            name = tags.get('name')
            
            if name:
                # Pour les surfaces (way/rel), les coordonnées sont dans 'center'
                lat = element.get('lat') or element.get('center', {}).get('lat')
                lon = element.get('lon') or element.get('center', {}).get('lon')
                
                spots.append({
                    'nom_lieu': name,
                    'ville': tags.get('addr:city') or tags.get('is_in:city') or "Hérault",
                    'categorie': tags.get('sport') or tags.get('leisure') or tags.get('natural') or 'lieu',
                    'lat': lat,
                    'lon': lon
                })
        
        return pd.DataFrame(spots)
    except Exception as e:
        print(f"Erreur : {e}")
        return pd.DataFrame()

df_complet = download_herault_spots_full()
# Nettoyage : on enlève les doublons de noms
df_complet = df_complet.drop_duplicates(subset=['nom_lieu'])

print(f"Nombre de lieux trouvés : {len(df_complet)}")
df_complet.to_csv("referentiel_lieux_herault.csv", index=False)

Erreur : 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter?data=%0A++++%5Bout%3Ajson%5D%5Btimeout%3A180%5D%3B%0A++++area%283600007411%29-%3E.searchArea%3B%0A++++%28%0A++++++nwr%5B%22sport%22%5D%28area.searchArea%29%3B%0A++++++nwr%5B%22leisure%22~%22park%7Cpitch%7Cgarden%7Cbeach_resort%7Cnature_reserve%7Cmarina%22%5D%28area.searchArea%29%3B%0A++++++nwr%5B%22tourism%22~%22viewpoint%7Cpicnic_site%22%5D%28area.searchArea%29%3B%0A++++++nwr%5B%22natural%22~%22peak%7Cbeach%7Cwater%22%5D%28area.searchArea%29%3B%0A++++%29%3B%0A++++out+center%3B%0A++++
Nombre de lieux trouvés : 0


In [10]:
df_complet

,nom_lieu,ville,categorie,lat,lon
0,Tour-observatoire,Hérault,lieu,49.447225,3.788529
1,Cascade de Saint-Martin Rivière,Hérault,lieu,50.042980,3.538022
2,La Hottée du Diable,Hérault,stone,49.182597,3.443871
3,Centre équestre,Hérault,equestrian,49.047360,3.500999
4,Le Dôme,Hérault,swimming,49.567233,3.651594
...,...,...,...,...,...
401,Réserve naturelle des Landes de Versigny,Hérault,nature_reserve,49.639708,3.461491
402,Réserve naturelle des prairies humides de la f...,Hérault,nature_reserve,49.888087,4.246229
403,Stade Cambreling,Hérault,running,49.929188,4.084929
404,Le parc d'Isle,Hérault,park,49.845459,3.310793


In [3]:
sample1 = [
    {'id' : 1, 'fingerprint': 1},
    {'id' : 2, 'fingerprint': 2},
    {'id' : 3, 'fingerprint': 3},
    {'id' : 4, 'fingerprint': 4},

]
sample2 = [
    {'id' : 1, 'fingerprint': 1},
    {'id' : 2, 'fingerprint': 3},
    {'id' : 3, 'fingerprint': 3},
    {'id' : 5, 'fingerprint': 5},

]
df1 = pd.DataFrame(sample1)
df2 = pd.DataFrame(sample2)
df3 = df1[['id', 'fingerprint']].merge(
            df2[['id', 'fingerprint']], 
            on=['id', 'fingerprint'], 
            how='outer', 
            indicator=True
        )
df3

,id,fingerprint,_merge
0,1,1,both
1,2,2,left_only
2,2,3,right_only
3,3,3,both
4,4,4,left_only
5,5,5,right_only


In [18]:
select = df3.loc[df3['id']==3].reset_index(drop=True)
str(select.loc[0].to_dict())

"{'id': 3, 'fingerprint': 3, '_merge': 'both'}"

In [3]:
ids_modifies = df3.loc[
    df3.duplicated(subset=['id'], keep=False), 'id'
].unique()

# 2. On crée le masque pour récupérer ces lignes dans le DataFrame de comparaison
mask_modifications = df3['id'].isin(ids_modifies)

# 3. On extrait les données
df_modifies = df3[mask_modifications]
df_modifies

,id,fingerprint,_merge
1,2,2,left_only
2,2,3,right_only


In [37]:
mask = df3['_merge'] == 'both'
df3[~mask]

,id,fingerprint,_merge
1,2,2,left_only
2,2,3,right_only
4,4,4,left_only
5,5,5,right_only


In [43]:
df3[df3['_merge'] == 'left_only']['id'].count()


np.int64(2)

df3[mask]